In [31]:

import osmnx as ox
import networkx as nx
import pandas as pd
import itertools
import os
from shapely.geometry import LineString

ox.settings.log_console = True
ox.settings.use_cache = True
FILENAME = "kocaeli_highways_only.graphml" 

DEFAULT_SPEEDS = {
    'motorway': 130,
    'motorway_link': 80,
    'trunk': 110,
    'trunk_link': 60,
    'primary': 90,
    'primary_link': 50,
    'secondary': 80, 
    'secondary_link': 40,
    'default': 50
}

DEFAULT_LANES = {
    'motorway': 3,
    'motorway_link': 1,
    'trunk': 2,     
    'trunk_link': 1,
    'primary': 2,
    'primary_link': 1,
    'secondary': 1,    
    'secondary_link': 1,
    'default': 1
}

def speed(speed_val, road_type):
    def get_default():
        rtype = 'default' 
        if isinstance(road_type, (list, tuple)):
            if len(road_type) > 0: rtype = road_type[0]
        elif isinstance(road_type, str):
            rtype = road_type
        return DEFAULT_SPEEDS.get(rtype, DEFAULT_SPEEDS['default']), 'varsayilan'

    if isinstance(speed_val, (list, tuple)):
        if len(speed_val) > 0:
            speed_val = speed_val[0]
        else:
            return get_default()

    if speed_val is None or pd.isna(speed_val) or speed_val == "":
        return get_default()


    try:
        if isinstance(speed_val, (int, float)):
            return int(speed_val), 'osm' 

        clean_str = ''.join(filter(str.isdigit, str(speed_val)))
        
        if not clean_str:
            return get_default()
            
        return int(clean_str), 'osm'

    except (ValueError, TypeError):
        return get_default()

def lanes(lane_val, road_type):
    def get_default():
        rtype = 'default'
        if isinstance(road_type, (list, tuple)):
            if len(road_type) > 0: rtype = road_type[0]
        elif isinstance(road_type, str):
            rtype = road_type
        return DEFAULT_LANES.get(rtype, DEFAULT_LANES['default']), 'varsayilan'

    if isinstance(lane_val, (list, tuple)):
        lane_val = lane_val[0] if len(lane_val) > 0 else None

    if lane_val is None or pd.isna(lane_val) or lane_val == "":
        return get_default()

    try:
        val_str = str(lane_val)
        
        if ';' in val_str:
             parts = [int(s) for s in val_str.split(';') if s.isdigit()]
             if parts:
                 return max(parts), 'osm' 
             else:
                 return get_default()
        
        clean_str = ''.join(filter(str.isdigit, val_str))
        
        if clean_str:
            return int(clean_str), 'osm' 
        else:
            return get_default()
            
    except (ValueError, TypeError):
        return get_default()
   
def traffic_light(node_id, G, road_type_at_segment):
   
    node_data = G.nodes[node_id]
    rtype_str = str(road_type_at_segment)

    if 'highway' in node_data:
        hw = node_data['highway']
        if isinstance(hw, list):
            if 'traffic_signals' in hw: return 1
        elif hw == 'traffic_signals':
            return 1

    if 'motorway' in rtype_str:
        return 0
        
    if 'link' in rtype_str:
        return 0

    if 'junction' in node_data:
        junc = node_data['junction']
        if 'roundabout' in str(junc):
            return 1

    try:
        degree = G.degree[node_id]
        
        if degree >= 4:
            return 1
            
    except:
        pass
        
    return 0
def smooth_speeds(df):
   
    df['prev_speed'] = df['max_speed'].shift(1)
    df['next_speed'] = df['max_speed'].shift(-1)
    df['prev_trip'] = df['trip_id'].shift(1)
    df['next_trip'] = df['trip_id'].shift(-1)
    
    
    for i in range(1, len(df) - 1):
       if df.at[i, 'speed_source'] == 'osm':

            continue
        
        
       if (df.at[i, 'trip_id'] == df.at[i, 'prev_trip']) and \
           (df.at[i, 'trip_id'] == df.at[i, 'next_trip']):
            
            current_speed = df.at[i, 'max_speed']
            prev_s = df.at[i, 'prev_speed']
            next_s = df.at[i, 'next_speed']
            

            if abs(prev_s - current_speed) > 30 or \
               (df.at[i, 'speed_source'] == 'varsayilan' and abs(prev_s - current_speed) > 30):
                
                new_speed = (prev_s + next_s + current_speed) / 3
                df.at[i, 'max_speed'] =int(5 * round(float(new_speed) / 5))
                df.at[i, 'speed_source'] = 'yumusatilmis' 

    df.drop(columns=['prev_speed', 'next_speed', 'prev_trip', 'next_trip'], inplace=True)
    return df


def get_filtered_graph():
  
    if os.path.exists(FILENAME):
        G = ox.load_graphml(FILENAME)
    else:
        
        custom_filter = '["highway"~"motorway|motorway_link|trunk|trunk_link|primary|primary_link|secondary|secondary_link"]'
        
        G = ox.graph_from_place(
            "Kocaeli, Turkey",
            network_type='drive',
            custom_filter=custom_filter
        )
        
        G = ox.add_edge_speeds(G)
        G = ox.add_edge_travel_times(G)
        
        ox.save_graphml(G, FILENAME)
        
    return G

def calculate_metrics(G):
    
    node_degrees = dict(G.degree())
    nx.set_node_attributes(G, node_degrees, 'node_degree')
    
    closeness = nx.closeness_centrality(G, distance='length')
    nx.set_node_attributes(G, closeness, 'node_closeness')

    node_betweenness = nx.betweenness_centrality(G, weight='length', normalized=True, k=3347)
    nx.set_node_attributes(G, node_betweenness, 'node_betweenness')

    centrality = nx.edge_betweenness_centrality(G, normalized=True, weight='length', k=3347)
    vals = list(centrality.values())
    if vals:
        min_val = min(vals)
        max_val = max(vals)
        
        
        if max_val - min_val > 0:
            for k in centrality:
                centrality[k] = ((centrality[k] - min_val) / (max_val - min_val)) * 100
        else:
            for k in centrality:
                centrality[k] = 50.0
    nx.set_edge_attributes(G, centrality, 'rarity_score')        
    return G

def generate_dataset(G):
    districts = {
        'Izmit': (40.7654, 29.9408),
        'Gebze': (40.8028, 29.4307),
        'Korfez': (40.7675, 29.7822),
        'Golcuk': (40.7183, 29.8242),
        'Kartepe': (40.7533, 30.0225),
        'Kandira': (41.0706, 30.1506),
        'Basiskele': (40.7126, 29.9298),
        'Derince': (40.7563, 29.8306),
        'Dilovasi': (40.7876, 29.5447),
        'Cayirova': (40.8166, 29.3777)
    }
    
    
    district_nodes = {k: ox.nearest_nodes(G, v[1], v[0]) for k, v in districts.items()}
    
    
    trips = list(itertools.permutations(district_nodes.keys(), 2))
    
    dataset = []
    
    trip_counter = 1    
    
    
    for orig, dest in trips:
        u_node = district_nodes[orig]
        v_node = district_nodes[dest]
        current_trip_name = f"{orig} - {dest}"
        try:
            
            path = nx.shortest_path(G, u_node, v_node, weight='travel_time')
            seq_counter = 1
            
            
            for u, v in zip(path[:-1], path[1:]):
                
                edge_data = G.get_edge_data(u, v)[0]
                custom_id = (trip_counter * 1000) + seq_counter
                osm_id = f"{u}-{v}"
                rtype = edge_data.get('highway', 'unknown')
                
                if isinstance(rtype, list):
                    rtype = rtype[0]
                
                raw_speed = edge_data.get('maxspeed', None)
                clean_speed, speed_source =speed(raw_speed, rtype)
                
                raw_lanes = edge_data.get('lanes', None)
                clean_lanes , lanes_source = lanes(raw_lanes, rtype)
                
                raw_tunnel = edge_data.get('tunnel', None)
                is_tunnel = 0
                
                if raw_tunnel == 'yes' or (isinstance(raw_tunnel, list) and 'yes' in raw_tunnel):
                    is_tunnel = 1
                
                has_light = traffic_light(v, G, rtype)
                
                degree = G.nodes[v]['node_degree'] 
                
                rarity = edge_data.get('rarity_score', 0)
                
                n_between = G.nodes[v].get('node_betweenness', 0)
                n_close = G.nodes[v].get('node_closeness', 0)

                
                if 'geometry' in edge_data:
                    coords = edge_data['geometry']
                else:
                    u_x, u_y = G.nodes[u]['x'], G.nodes[u]['y']
                    v_x, v_y = G.nodes[v]['x'], G.nodes[v]['y']
                    coords = LineString([(u_x, u_y), (v_x, v_y)])
                                
                row = {
                    'trip_id': trip_counter,
                    'trip_name': current_trip_name,
                    'segment_id': custom_id,
                    'osm_segment_id': osm_id,
                    'length_meters': edge_data.get('length', 0),
                    'road_type': rtype,
                    'lanes': clean_lanes,
                    'lanes_source': lanes_source,
                    'tunnel': is_tunnel,
                    'max_speed': clean_speed,
                    'speed_source': speed_source,
                    'traffic_light': has_light,
                    'coords': coords,
                    'node_degree': degree,
                    'betweenness': n_between,  
                    'closeness': n_close,      
                    'rarity_score': rarity
                }
                dataset.append(row)
            
                seq_counter += 1
            
            trip_counter += 1
            
        except nx.NetworkXNoPath:
            continue

    return pd.DataFrame(dataset)


G = get_filtered_graph()

G = calculate_metrics(G)

df = generate_dataset(G)

final_columns = [
    'trip_id', 
    'trip_name',
    'segment_id',
    'osm_segment_id',
    'length_meters', 
    'road_type', 
    'lanes',
    'lanes_source',
    'tunnel',
    'max_speed',
    'speed_source',
    'traffic_light',
    'coords', 
    'node_degree', 
    'betweenness',
    'closeness',
    'rarity_score'
]
df = df[final_columns]
df = smooth_speeds(df)       

df.to_csv("GNN_DATA.csv", index=False)